In [1]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader

from typing import Annotated
from pydantic import BaseModel,Field

from dotenv import load_dotenv

C:\Users\SUBHAM\AppData\Local\Temp\ipykernel_5364\1901615858.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
load_dotenv()

emb_model=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

llm_model=ChatGroq(model="openai/gpt-oss-120b")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
from pydantic import BaseModel, Field
from typing import List, Optional


class ResumeAnalysis_Schema(BaseModel):
    name: str = Field(description="Full name of the candidate")

    email: Optional[str] = Field( default=None,description="Candidate's email address")

    education: List[str] = Field( description="Educational qualifications" )

    experience: List[str] = Field( description="Work experience or internships")

    skills: List[str] = Field( description="Technical and professional skills")

    projects: List[str] = Field(  description="Projects completed by the candidate" )

    strengths: List[str] = Field(  description="Strong points identified in the resume")

    weaknesses: List[str] = Field(  description="Areas that need improvement")

    ats_score: int = Field( ge=0, le=100, description="ATS compatibility score between 0 and 100")

    missing_skills: List[str] = Field(description="Important skills missing for the candidate's target role" )

    improvement_suggestions: List[str] = Field( description="Suggestions to improve the resume")

    summary: str = Field( description="Overall professional summary of the candidate")

In [5]:
llm_with_schema=llm_model.with_structured_output(ResumeAnalysis_Schema)

In [11]:
loader=PyPDFLoader("Resume_store_folder/subham padhi resume.pdf")

In [19]:
document=loader.load()

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

In [21]:
len(document)

1

In [22]:
chunks=splitter.split_documents(document)

In [23]:
len(chunks)

13

In [24]:
vector_store=FAISS.from_documents(
    chunks,emb_model
)

In [25]:
retriver=vector_store.as_retriever(
   search_type="similarity",
   search_kwargs={"k":3}
)

In [43]:
from langchain_core.prompts import PromptTemplate

parser=StrOutputParser()


prompt = PromptTemplate(
    template="""
You are an expert Resume Analyzer and ATS specialist.

Analyze the following resume accurately and provide the required structured information.

Do not invent information. If information is unavailable, use an empty list or null.
Keep the analysis professional and concise.

Resume:
{content}

""",
    input_variables=["content"]
)

In [47]:
chain = (
    {"content": retriver | (lambda docs: "\n\n".join(d.page_content for d in docs))}
    | prompt
    | llm_with_schema
)
result = chain.invoke("Summarize this resume")

In [48]:
result

ResumeAnalysis_Schema(name='Subham Padhi', email='subham117700@gmail.com', education=['B.Tech in Computer Science Engineering (AI & ML), GITA Autonomous College, Bhubaneswar, CGPA: 9.48/10 (2024-2028, till 5th semester)'], experience=['Artificial Intelligence Intern, XTRAGRAD Pvt. Ltd. (Jul 2026 - Aug 2026): Backend Team Member contributing to development of AI CareerOS, an AI-powered platform integrating learning, career development, and recruitment.'], skills=['Python', 'Machine Learning', 'Deep Learning', 'Natural Language Processing (NLP)', 'Generative AI', 'FastAPI', 'SQL', 'Backend Development'], projects=[], strengths=['High academic performance (CGPA 9.48/10).', 'Hands‑on AI internship experience with backend development.', 'Proficiency in core AI/ML concepts and Python ecosystem.', 'Familiarity with API development using FastAPI.'], weaknesses=['Limited professional experience beyond a short internship.', 'No listed projects or portfolio items to demonstrate applied skills.', 